# Experiment: Case-001 Registry Link Disambiguation Gate

Objective: classify registry links before owner routing or case-specific interpretation.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

tdt_query = json.loads(
    Path(
        "data/registries/clinicaltrials/2026-05-09-active-tdt-registry-refresh.json"
    ).read_text()
)
mitapivat_tdt = json.loads(
    Path(
        "data/registries/clinicaltrials/2026-05-10-nct07506863-mitapivat-pediatric-tdt-detail.json"
    ).read_text()
)

records = [*tdt_query["records"], *mitapivat_tdt["records"]]


def classify(title: str) -> str:
    """Return a public-safe registry context label from a study title."""
    text = title.lower()
    if "iron overload" in text:
        return "query_adjacent_iron_axis"
    if "long-term follow-up" in text:
        return "follow_up_context"
    if "non-transfusion" in text:
        return "query_adjacent_ntdt"
    if "bcl11a" in text or "gene editing" in text:
        return "gene_cell_context"
    if "transfusion-dependent" in text or "tdt" in text:
        return "direct_tdt_context"
    if "transplant" in text or "undergoing ucbt" in text:
        return "broad_transplant_context"
    return "unclear_context"


classes = {record["nct_id"]: classify(record["title"]) for record in records}
decision = {
    "gate_label": "registry_link_disambiguation_ready",
    "records_checked": len(records),
    "classes": classes,
    "policy": "registry_presence_is_not_case_eligibility",
}

decision

## Decision

Continue with registry-link disambiguation. Public trial links must be classified before private owner routing.